# ViMD – Kiểm tra khả thi (feasibility check)

Notebook này trả lời 4 câu hỏi trước khi chốt đề tài:
1. Tải và đọc được ViMD không, dữ liệu nặng bao nhiêu?
2. **Số người nói mỗi tỉnh** có đủ để chia train/test **không trùng người nói** không? Split chính thức có bị trùng người nói không?
3. Đặc trưng từ mô hình pre-trained (XLS-R) phân biệt được 63 tỉnh tốt hơn ngẫu nhiên bao nhiêu? Tầng nào tốt nhất?
4. Nếu để trùng người nói giữa train/test thì kết quả bị "ảo" bao nhiêu?

**Cách chạy (Kaggle):** Settings → Accelerator = **GPU T4**, Internet = **On** → *Run All*. Kết thúc sẽ có file `feasibility_report.json` trong tab Output.

In [ ]:
# ==== CẤU HÌNH (thường không cần sửa) ====
REPO_ID = "nguyendv02/ViMD_Dataset"
MODEL_NAME = "facebook/wav2vec2-xls-r-300m"   # mô hình pre-trained dùng để trích đặc trưng
DATA_DIR = None        # None = tự chọn thư mục còn trống nhiều nhất
PROVINCE_COL = None    # None = tự dò; nếu dò sai, điền tên cột ở đây
SPEAKER_COL = None
REGION_COL = None
N_TRAIN_PER_PROV = 30  # số câu mỗi tỉnh dùng để train probe
N_TEST_PER_PROV = 10   # số câu mỗi tỉnh dùng để test
MAX_SEC = 8.0          # cắt mỗi audio tối đa 8 giây cho nhanh
SEED = 42

## 1. Tải dữ liệu

In [ ]:
import os, io, glob, json, time, shutil, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()
report = {}

def pick_dir():
    cands = ["/kaggle/tmp", "/tmp", "/content", os.getcwd()]
    ok = [d for d in cands if os.path.isdir(d) and os.access(d, os.W_OK)]
    return os.path.join(max(ok, key=lambda d: shutil.disk_usage(d).free), "vimd")

DATA_DIR = DATA_DIR or pick_dir()
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| dung lượng trống: %.1f GB" % (shutil.disk_usage(DATA_DIR).free / 1e9))

token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not glob.glob(f"{DATA_DIR}/**/*.parquet", recursive=True):
    from huggingface_hub import snapshot_download
    t = time.time()
    snapshot_download(REPO_ID, repo_type="dataset", local_dir=DATA_DIR,
                      allow_patterns=["*.parquet", "*.md"], token=token, max_workers=8)
    report["download_minutes"] = round((time.time() - t) / 60, 1)
    print("Tải xong sau %.1f phút" % report["download_minutes"])

files = sorted(glob.glob(f"{DATA_DIR}/**/*.parquet", recursive=True))
report["n_parquet_files"] = len(files)
report["dataset_gb"] = round(sum(os.path.getsize(f) for f in files) / 1e9, 2)
print(len(files), "file parquet,", report["dataset_gb"], "GB")
assert files, "Không thấy file parquet nào – xem lại lỗi tải ở trên."


## 2. Đọc metadata (không đọc audio) và dò cột

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq

def split_of(path):
    name = os.path.basename(path).lower()
    parts = [name] + [p.lower() for p in path.split(os.sep)[-3:-1]]
    for key, val in [("train", "train"), ("valid", "valid"), ("dev", "valid"), ("test", "test")]:
        if any(p.startswith(key) or p == key for p in parts):
            return val
    return "unknown"

schema = pq.read_schema(files[0])
print(schema)

def is_audio(field):
    t = field.type
    if pa.types.is_struct(t):
        names = [t.field(i).name for i in range(t.num_fields)]
        return "bytes" in names or "array" in names
    return False

AUDIO_COL = next((f.name for f in schema if is_audio(f)), None)
assert AUDIO_COL, "Không tìm thấy cột audio – gửi lại phần schema in ở trên."
meta_cols = [f.name for f in schema if f.name != AUDIO_COL]

dfs = []
for f in files:
    d = pq.read_table(f, columns=meta_cols).to_pandas()
    d["_file"], d["_row"], d["_split"] = f, np.arange(len(d)), split_of(f)
    dfs.append(d)
meta = pd.concat(dfs, ignore_index=True)

def find_col(keys, given):
    if given:
        return given
    for k in keys:
        for c in meta.columns:
            if k in c.lower() and not c.startswith("_"):
                return c
    return None

PROVINCE_COL = find_col(["province", "tinh", "dialect"], PROVINCE_COL)
SPEAKER_COL = find_col(["speaker", "spk"], SPEAKER_COL)
REGION_COL = find_col(["region", "mien"], REGION_COL)
DUR_COL = find_col(["duration", "length"], None)
print("\nCột phát hiện:", dict(audio=AUDIO_COL, province=PROVINCE_COL, speaker=SPEAKER_COL,
                                region=REGION_COL, duration=DUR_COL))
print(meta.drop(columns=["_file"]).head(3).T)
assert PROVINCE_COL, "Không dò được cột tỉnh – điền PROVINCE_COL ở ô cấu hình rồi chạy lại."
report["columns"] = dict(all=meta_cols, audio=AUDIO_COL, province=PROVINCE_COL,
                         speaker=SPEAKER_COL, region=REGION_COL, duration=DUR_COL)


## 3. Thống kê: số tỉnh, số người nói, trùng người nói giữa các split

In [ ]:
P = PROVINCE_COL
meta[P] = meta[P].astype(str)
if SPEAKER_COL:
    # ghép với tỉnh để tránh trường hợp mã người nói chỉ duy nhất trong từng tỉnh
    meta["_spk"] = meta[P] + "|" + meta[SPEAKER_COL].astype(str)

report["n_utterances"] = int(len(meta))
report["utt_per_split"] = meta["_split"].value_counts().to_dict()
report["n_provinces"] = int(meta[P].nunique())
upp = meta.groupby(P).size()
report["utt_per_province"] = dict(min=int(upp.min()), median=float(upp.median()), max=int(upp.max()))
if DUR_COL and pd.api.types.is_numeric_dtype(meta[DUR_COL]):
    report["hours_per_split"] = (meta.groupby("_split")[DUR_COL].sum() / 3600).round(2).to_dict()

if SPEAKER_COL:
    spp = meta.groupby(P)["_spk"].nunique().sort_values()
    report["n_speakers"] = int(meta["_spk"].nunique())
    report["speakers_per_province"] = dict(min=int(spp.min()), median=float(spp.median()), max=int(spp.max()))
    report["provinces_with_le2_speakers"] = {k: int(v) for k, v in spp[spp <= 2].items()}
    sets = {s: set(g["_spk"]) for s, g in meta.groupby("_split")}
    names = sorted(sets)
    report["official_split_speaker_overlap"] = {
        f"{a}&{b}": len(sets[a] & sets[b]) for i, a in enumerate(names) for b in names[i + 1:]}
    print("Số người nói ít nhất/tỉnh:", spp.head(10).to_dict())
else:
    print("CẢNH BÁO: không có cột người nói -> không kiểm tra được rò rỉ người nói.")

print(json.dumps({k: v for k, v in report.items() if k != "columns"}, ensure_ascii=False, indent=1))


## 4. Chia train/test không trùng người nói và chọn mẫu

In [ ]:
rng = np.random.default_rng(SEED)
if SPEAKER_COL:
    train_spk, test_spk, skipped = set(), set(), []
    for p, g in meta.groupby(P):
        spks = np.array(sorted(g["_spk"].unique()))
        rng.shuffle(spks)
        if len(spks) < 2:
            skipped.append(p); continue
        n_test = max(1, int(round(len(spks) * 0.25)))
        test_spk.update(spks[:n_test]); train_spk.update(spks[n_test:])
    pool_tr = meta[meta["_spk"].isin(train_spk)]
    pool_te = meta[meta["_spk"].isin(test_spk)]
    report["split_used"] = "speaker-disjoint (tự chia)"
    report["provinces_skipped_lt2_speakers"] = skipped
else:
    pool_tr = meta[meta["_split"] == "train"]
    pool_te = meta[meta["_split"] == "test"]
    report["split_used"] = "official (KHÔNG kiểm tra được người nói)"

def sample(pool, n):
    return pool.sample(frac=1, random_state=SEED).groupby(P).head(n)

sel_tr = sample(pool_tr, N_TRAIN_PER_PROV).assign(_set="train")
sel_te = sample(pool_te, N_TEST_PER_PROV).assign(_set="test")
sel = pd.concat([sel_tr, sel_te], ignore_index=True)
print(report["split_used"], "| train:", len(sel_tr), "câu, test:", len(sel_te), "câu,",
      sel[P].nunique(), "tỉnh")


## 5. Trích đặc trưng bằng XLS-R (mean-pool từng tầng)

In [ ]:
import torch, soundfile as sf, librosa, tempfile
from transformers import AutoFeatureExtractor, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
fe = AutoFeatureExtractor.from_pretrained(MODEL_NAME, token=token)
model = AutoModel.from_pretrained(MODEL_NAME, token=token).to(device).eval()
SR = 16000

def decode(cell):
    if cell.get("array") is not None:
        y, sr = np.asarray(cell["array"], dtype=np.float32), cell.get("sampling_rate") or SR
    else:
        b = cell["bytes"]
        try:
            y, sr = sf.read(io.BytesIO(b), dtype="float32")
        except Exception:
            ext = os.path.splitext(cell.get("path") or "")[1] or ".mp3"
            with tempfile.NamedTemporaryFile(suffix=ext) as tmp:
                tmp.write(b); tmp.flush()
                y, sr = librosa.load(tmp.name, sr=None, mono=True)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y[: int(MAX_SEC * SR)]

@torch.no_grad()
def embed(y):
    x = fe(y, sampling_rate=SR, return_tensors="pt").input_values.to(device)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        hs = model(x, output_hidden_states=True).hidden_states
    return torch.stack([h.float().mean(dim=1)[0] for h in hs]).cpu().numpy()  # (n_layers+1, dim)

feats, keep, audio_sec, fails = [], [], 0.0, 0
t = time.time()
for i, (f, g) in enumerate(sel.groupby("_file")):
    col = pq.read_table(f, columns=[AUDIO_COL]).column(0)
    for idx, r in g.iterrows():
        try:
            y = decode(col[int(r["_row"])].as_py())
            if len(y) < SR // 2:
                fails += 1; continue
            feats.append(embed(y)); keep.append(idx); audio_sec += len(y) / SR
        except Exception as e:
            fails += 1
            if fails <= 3: print("Lỗi đọc audio:", repr(e)[:200])
    if i % 10 == 0:
        print(f"  {len(feats)}/{len(sel)} câu, {time.time() - t:.0f}s")

el = time.time() - t
X = np.stack(feats); S = sel.loc[keep].reset_index(drop=True)
report["embed"] = dict(n_ok=len(feats), n_fail=fails, seconds=round(el, 1),
                       sec_per_clip=round(el / max(1, len(feats)), 3),
                       est_minutes_full_dataset=round(el / max(1, len(feats)) * len(meta) / 60, 1),
                       device=str(device), feature_shape=list(X.shape))
print(report["embed"])


## 6. Probing từng tầng: phân loại 63 tỉnh bằng Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from sklearn.model_selection import train_test_split

def probe(Xtr, ytr, Xte, yte):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.1))
    clf.fit(Xtr, ytr)
    proba = clf.predict_proba(Xte); classes = clf.classes_
    pred = classes[proba.argmax(1)]
    k = min(5, len(classes) - 1)
    return dict(acc=accuracy_score(yte, pred), macro_f1=f1_score(yte, pred, average="macro"),
                top5=top_k_accuracy_score(yte, proba, k=k, labels=classes)), pred

tr, te = (S["_set"] == "train").values, (S["_set"] == "test").values
y = S[P].values
rows = []
for l in range(X.shape[1]):
    m, _ = probe(X[tr, l], y[tr], X[te, l], y[te])
    rows.append(dict(layer=l, **{k: round(v, 4) for k, v in m.items()}))
    print(rows[-1])
layers = pd.DataFrame(rows)
best = int(layers.loc[layers["macro_f1"].idxmax(), "layer"])
report["chance_acc"] = round(1 / S[P].nunique(), 4)
report["layer_probe"] = rows
report["best_layer"] = rows[best]
print("\nTầng tốt nhất:", rows[best], "| ngẫu nhiên ≈", report["chance_acc"])


## 7. Kiểm tra rò rỉ người nói và độ chính xác theo vùng

In [ ]:
# (a) Rò rỉ: trộn lại cùng tập câu nhưng chia ngẫu nhiên theo câu -> người nói xuất hiện ở cả train và test
if SPEAKER_COL:
    idx_tr, idx_te = train_test_split(np.arange(len(S)), test_size=te.mean(), stratify=y, random_state=SEED)
    leak, _ = probe(X[idx_tr, best], y[idx_tr], X[idx_te, best], y[idx_te])
    shared = len(set(S.loc[idx_tr, "_spk"]) & set(S.loc[idx_te, "_spk"]))
    report["leaky_random_split"] = dict(**{k: round(v, 4) for k, v in leak.items()}, shared_speakers=shared)
    print("Chia ngẫu nhiên theo câu (có rò rỉ):", report["leaky_random_split"])
    print("Chia không trùng người nói          :", rows[best])

# (b) Theo vùng: đoán đúng vùng miền dù sai tỉnh?
if REGION_COL:
    p2r = meta.groupby(P)[REGION_COL].agg(lambda s: s.mode().iloc[0]).astype(str).to_dict()
    _, pred = probe(X[tr, best], y[tr], X[te, best], y[te])
    report["region_acc_from_province_pred"] = round(float(np.mean(
        [p2r[a] == p2r[b] for a, b in zip(y[te], pred)])), 4)
    print("Đúng vùng (suy từ dự đoán tỉnh):", report["region_acc_from_province_pred"])

report["total_minutes"] = round((time.time() - T0) / 60, 1)
with open("feasibility_report.json", "w", encoding="utf-8") as fp:
    json.dump(report, fp, ensure_ascii=False, indent=1, default=str)
print("\nĐã lưu feasibility_report.json – gửi file này lại cho Claude.")
